# Create Graph Dataset
We showcase how to turn SMILES into graphs and save it, just like our QM9 dataset.

In [1]:
!pip install -q rdkit torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.2 MB/s eta 0:00:00


In [2]:
import torch
from torch_geometric.data import InMemoryDataset, Data
from rdkit import Chem
from rdkit.Chem import AllChem

In [3]:
# Atom feature function, we only include atomic numbers
def atom_features(atom):
    # One-hot for atom type (H, C, N, O, F)
    atom_types = ['H','C','N','O','F']
    a = [atom.GetSymbol() == x for x in atom_types]
    return torch.tensor(a, dtype=torch.float) # turn into torch.tensor()


In [4]:
# Turn rdkit molecule to graph object
def mol_to_graph(mol):

    # first create the node feature
    N = mol.GetNumAtoms()
    x = torch.stack([atom_features(atom) for atom in mol.GetAtoms()])

    # Edge indices
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i,j])
        edge_index.append([j,i])  # undirected graph

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Optional: edge features (bond type)
    # Can add bond type one-hot here if desired

    data = Data(x=x, edge_index=edge_index) # graph data format
    return data


## Class to turn SMILES into graphs that are compatible with PyTorch Geometric

In [5]:
class MyMoleculeDataset(InMemoryDataset):
    def __init__(self, root, smiles_list=None, transform=None, pre_transform=None, target_fn=None,):
        '''
        InMemoryDataset is a class to load all dataset into memory at once.
        If you do not have large memory or too big dataset, avoid using this option.

        Args:
          root: root directory of the dataset
          smiles_list: list of SMILES strings
          transform: transform function to apply to each graph
          pre_transform: transform function to apply to the whole dataset
          target_fn: function to apply to each graph to get the target
        '''
        self.smiles_list = smiles_list
        self.target_fn = target_fn
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False) # data is the data itself, slices stores the look up table for each graph

    @property
    def raw_file_names(self):
        return []  # Not used
    # the symbol @property can allow a function to behave like an attribute
    # you may use mydataset.raw_file_names instead of calling mydataset.raw_file_names()

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        pass  # Not needed

    def process(self):
        data_list = []
        for smi in self.smiles_list:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            # mol = Chem.AddHs(mol) # if you want H then uncomment these two lines
            # AllChem.EmbedMolecule(mol)
            data = mol_to_graph(mol)
            data_list.append(data)

            # Apply target function if provided
            if self.target_fn is not None:
                data.y = self.target_fn(data)

        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])


## Now let's use it

In [6]:
smiles = ['CCO', 'CC', 'CCN', 'C1CCCCC1']

# optional
def example_target_func(data):
    # returns the number of heavy atoms
    # assume data.x one-hot: [H,C,N,O,F]
    atom_types = ['H','C','N','O','F']
    # count all non-H atoms
    is_heavy = 1 - data.x[:,0]  # 0=H, 1-heavy
    return torch.tensor([is_heavy.sum()], dtype=torch.float)

dataset = MyMoleculeDataset(root='mymol', smiles_list=smiles, target_fn=example_target_func)

print(dataset[0])  # PyG Data object
print(dataset[0].x.shape)
print(dataset[0].edge_index.shape)

Data(x=[3, 5], edge_index=[2, 4], y=[1])
torch.Size([3, 5])
torch.Size([2, 4])


Processing...
Done!


### You can simply load your data by providing it's root folder

In [7]:
dataset = MyMoleculeDataset(root='mymol')